# Ontology And Ask Sage Row-Level Classification Pipeline

This notebook builds the raw CSV ontology, classifies each programming funding row against Genesis Mission Challenges and AI-directness with Ask Sage credentials from `credentials.json`, enriches the graph with evidence edges, and exports the validation tables used by the dashboard.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
import networkx as nx
from IPython.display import display

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")

PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from programming_ontology import paths
from programming_ontology.asksage_classifier import (
    AskSageAIDirectnessClassifier,
    AskSageChallengeClassifier,
    default_credentials_path,
    resolve_asksage_openai_base_url,
)
from programming_ontology.build_ontology import build_combined_ontology, export_graph, summarize_graph
from programming_ontology.classify_programming import (
    apply_sme_overrides,
    build_challenge_profiles,
    build_office_claims,
    build_programming_evidence_profiles,
    build_validation_outputs,
    candidate_rankings_to_dataframe,
    classify_programming_records,
    classification_results_to_dataframe,
    enrich_graph_with_reviewed_classifications,
    export_classifications,
    export_validation_outputs,
    load_or_initialize_sme_review,
)

paths.ensure_output_dirs()

In [ ]:
combined_graph = build_combined_ontology()
challenges = build_challenge_profiles(combined_graph)
office_claims = build_office_claims(combined_graph, challenges)
claimed_offices = {claim.claimed_office for claim in office_claims}
evidence_rows = build_programming_evidence_profiles(combined_graph, claimed_offices=claimed_offices)

print(summarize_graph(combined_graph))
print(f"Challenge candidates: {len(challenges)}")
print(f"Office claims: {len(office_claims)}")
print(f"Programming evidence rows: {len(evidence_rows)}")

In [ ]:
preview = candidate_rankings_to_dataframe(evidence_rows[:10], challenges, top_k=5)
display(preview.head(30))

In [ ]:
ASKSAGE_TOP_K = 3
ASKSAGE_REASONING_EFFORT = os.environ.get("ASKSAGE_REASONING_EFFORT", "medium").strip()
ASKSAGE_RESUME_CACHE = True
ASKSAGE_TIMEOUT_SECONDS = 120
ASKSAGE_CREDENTIALS_PATH = default_credentials_path()
ASKSAGE_MODEL = os.environ.get("ASKSAGE_MODEL", "").strip() or None
ASKSAGE_OPENAI_BASE_URL = resolve_asksage_openai_base_url(credentials_path=ASKSAGE_CREDENTIALS_PATH)

llm_classifier = AskSageChallengeClassifier(
    model=ASKSAGE_MODEL,
    reasoning_effort=ASKSAGE_REASONING_EFFORT,
    credentials_path=ASKSAGE_CREDENTIALS_PATH,
    audit_cache_path=paths.ASKSAGE_CLASSIFICATION_AUDIT_PATH,
    resume_cache=ASKSAGE_RESUME_CACHE,
    timeout_seconds=ASKSAGE_TIMEOUT_SECONDS,
)
ai_directness_classifier = AskSageAIDirectnessClassifier(
    model=ASKSAGE_MODEL,
    reasoning_effort=ASKSAGE_REASONING_EFFORT,
    credentials_path=ASKSAGE_CREDENTIALS_PATH,
    audit_cache_path=paths.ASKSAGE_AI_DIRECTNESS_AUDIT_PATH,
    resume_cache=ASKSAGE_RESUME_CACHE,
    timeout_seconds=ASKSAGE_TIMEOUT_SECONDS,
)

print(f"Using Ask Sage credentials file: {ASKSAGE_CREDENTIALS_PATH}")
print(f"Ask Sage model: {llm_classifier.model}")
print(f"Resolved OpenAI SDK base URL: {ASKSAGE_OPENAI_BASE_URL}")
print("Structured output uses credentials.json openai_url when provided; otherwise it derives /openai/v1 from the server URL.")
print("Running one-row structured-output smoke test...")

smoke_results = classify_programming_records(
    evidence_rows[:1],
    challenges,
    llm_classifier=llm_classifier,
    ai_directness_classifier=ai_directness_classifier,
    top_k=ASKSAGE_TOP_K,
)
smoke_df = classification_results_to_dataframe(smoke_results)
display(smoke_df[["programming_record_id", "predicted_challenge", "confidence", "ai_directness_score", "ai_directness_label", "rationale"]])

classification_results = classify_programming_records(
    evidence_rows,
    challenges,
    llm_classifier=llm_classifier,
    ai_directness_classifier=ai_directness_classifier,
    top_k=ASKSAGE_TOP_K,
)
classification_df = classification_results_to_dataframe(classification_results)
export_classifications(classification_results, paths.PROGRAMMING_CHALLENGE_CLASSIFICATIONS_PATH)
classification_df.to_csv(paths.CLASSIFICATION_DF_PATH, index=False)
display(classification_df.head(10))

In [ ]:
sme_review, created_review_file = load_or_initialize_sme_review(
    paths.SME_REVIEW_OVERRIDES_PATH,
    classification_df,
)
reviewed_classifications = apply_sme_overrides(classification_df, sme_review, challenges)
reviewed_classifications.to_csv(paths.REVIEWED_PROGRAMMING_CHALLENGE_CLASSIFICATIONS_PATH, index=False)

validation_outputs = build_validation_outputs(reviewed_classifications, office_claims)
export_validation_outputs(
    validation_outputs,
    {
        "office_claim_validation": paths.OFFICE_CLAIM_VALIDATION_PATH,
        "challenge_coverage_summary": paths.CHALLENGE_COVERAGE_SUMMARY_PATH,
        "office_challenge_evidence": paths.OFFICE_CHALLENGE_EVIDENCE_PATH,
        "wbs_challenge_summary": paths.WBS_CHALLENGE_SUMMARY_PATH,
    },
)
validation_outputs["challenge_coverage_summary"].to_csv(paths.REVIEW_SUMMARY_PATH, index=False)
print("Created review template." if created_review_file else "Loaded existing review template.")
for name, frame in validation_outputs.items():
    print(f"{name}: {frame.shape[0]} rows x {frame.shape[1]} columns")

In [ ]:
classified_graph = enrich_graph_with_reviewed_classifications(combined_graph, reviewed_classifications)
export_graph(combined_graph, paths.COMBINED_ONTOLOGY_GRAPH_PATH)
export_graph(classified_graph, paths.CLASSIFIED_ONTOLOGY_GRAPH_PATH)

relation_counts = pd.Series(
    [data.get("relation") for _, _, _, data in classified_graph.edges(keys=True, data=True)]
).value_counts()
display(relation_counts.to_frame("edge_count"))